## Governance at Scale
Role-based access control (RBAC) assigns privileges to roles and roles to users. This lab follows one customer table through role design, tagging, classification, column- and row-level security, and audit evidence.
> **Edition:** Enterprise Edition or higher is required for the complete path: data classification, column-level security, row-level security, and Access History.

In this hands-on lab you'll work through 6 sections:

| # | Section | Outcome |
|---|---------|---------|
| 1 | RBAC at Scale (Functional + Access + Database Roles) | layer roles so access grows without a grant-management nightmare |
| 2 | Object Tagging Strategy (DOMAIN, ENVIRONMENT) | make governed objects queryable by business metadata, not just name |
| 3 | Data Classification (Horizon Catalog Auto-Tag) | find and label sensitive columns automatically instead of by memory |
| 4 | Column-Level Security (Masking Policies) | protect sensitive columns without duplicating the table per audience |
| 5 | Row-Level Security (Row Access Policies) | restrict which rows a role can see, not just which columns |
| 6 | Access History for Audit | answer 'who touched this data and when' after the fact |

### Setup

Create the lab database and tag this session.

In [ ]:
-- Attribution: tag this session's queries (no privilege needed; role-agnostic)
ALTER SESSION SET QUERY_TAG = '{"origin":"sf_ace","name":"governance-at-scale","version":{"major":1,"minor":0},"attributes":{"program":"ace-webinar-series","webinar":"w06","source":"sql"}}';

USE ROLE SYSADMIN;
CREATE DATABASE IF NOT EXISTS GOVERNANCE_AT_SCALE_HOL;
USE DATABASE GOVERNANCE_AT_SCALE_HOL;
USE SCHEMA PUBLIC;
CREATE OR REPLACE WAREHOUSE GOVERNANCE_AT_SCALE_WH
  WAREHOUSE_SIZE = XSMALL
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Dedicated XS compute for this HOL; dropped in cleanup';
USE WAREHOUSE GOVERNANCE_AT_SCALE_WH;

## Section 1 — RBAC at Scale (Functional + Access + Database Roles)

**The problem:** This lab separates reusable object-access privileges from job-function roles. Inspect object grants and role membership together when reviewing access; there is no requirement to scan every user's grant list individually.


**[Role-Based Access Control (RBAC)](https://docs.snowflake.com/en/user-guide/security-access-control-overview)** assigns privileges to roles and roles to users. Snowflake recommends using logical **access roles** for object privileges and granting them to **functional roles** aligned with business functions; privileges on lower roles are inherited by roles above them. A **[database role](https://docs.snowflake.com/en/user-guide/security-access-control-overview#label-access-control-database-roles)** adds a separate scope boundary: it can hold privileges only within its database and must be granted to an account role before it can be active in a session. In this lab, one database role feeds one access role, which feeds one functional role.


> **Outcome:** layer roles so access grows without a grant-management nightmare.

### Core demo — Build one scalable access path

**Role-Based Access Control (RBAC)** separates job functions from reusable privilege bundles. We will create a functional role, an access role, and a database role, then wire the inheritance chain once.

**You should see:** `HOL_FUNCTIONAL_ROLE` inherits `HOL_ACCESS_ROLE`, while `HOL_DB_ROLE` remains scoped to the lab database.

> Docs: [Access control overview](https://docs.snowflake.com/en/user-guide/security-access-control-overview) · [Database roles](https://docs.snowflake.com/en/user-guide/security-access-control-overview#label-access-control-database-roles)

In [ ]:
-- Core path: pre-stage the role objects and wire the reusable access chain
USE ROLE USERADMIN;
CREATE OR REPLACE ROLE HOL_FUNCTIONAL_ROLE COMMENT = 'HOL job-function role';
CREATE OR REPLACE ROLE HOL_ACCESS_ROLE COMMENT = 'HOL reusable access bundle';

USE ROLE SECURITYADMIN;
GRANT ROLE HOL_ACCESS_ROLE TO ROLE HOL_FUNCTIONAL_ROLE;

USE ROLE SYSADMIN;
USE DATABASE GOVERNANCE_AT_SCALE_HOL;
CREATE OR REPLACE DATABASE ROLE GOVERNANCE_AT_SCALE_HOL.HOL_DB_ROLE
  COMMENT = 'HOL database-scoped role';

USE ROLE SECURITYADMIN;
GRANT DATABASE ROLE GOVERNANCE_AT_SCALE_HOL.HOL_DB_ROLE TO ROLE HOL_ACCESS_ROLE;

In [ ]:
-- Confirm: inspect the inheritance edge used throughout the demo
SHOW GRANTS TO ROLE HOL_FUNCTIONAL_ROLE;
SHOW GRANTS TO ROLE HOL_ACCESS_ROLE;
SHOW DATABASE ROLES IN DATABASE GOVERNANCE_AT_SCALE_HOL;

### Take-home — RBAC design variations

The core recording uses one inheritance path. After the session, use the detailed examples in the source manifest to explore one-step revocation, naming conventions, role-graph inspection, and production-safe database-role replacement. Grant roles, rather than object privileges, to users; map real job functions to functional roles and reusable privilege bundles to access or database roles.

> Docs: [Access control best practices](https://docs.snowflake.com/en/user-guide/security-access-control-considerations)

### Verify in Snowsight — Role Hierarchy Graph (read-only)

[Open in Snowsight](https://app.snowflake.com/_deeplink/#/account/roles/graph)

Read-only — confirm what you just built; all objects were created by the SQL above.

> **Apply on your account**
>
> In production, name functional roles after the business functions your organization actually uses, and name access roles after the object scope and access level they bundle. Snowflake recommends this logical separation because permissions granted to lower roles are inherited by roles above them.
>
> Use database roles when privileges should stay within one database. Grant the database role to an account role before using those privileges in a session. If a provider grants a database role to a share, each consumer still grants that shared database role to its own account roles.
>
> Treat `CREATE OR REPLACE DATABASE ROLE` as destructive replacement. Prefer a non-replacing creation or an explicit alteration when preserving existing grants matters.

## Section 2 — Object Tagging Strategy (DOMAIN, ENVIRONMENT)

**The problem:** When a data estate grows past a handful of tables, nobody can answer
"which tables belong to Finance, which are in Production, and who pays for what?"
without opening a spreadsheet — which is always out of date.


A **[tag](https://docs.snowflake.com/en/user-guide/object-tagging)** is a
schema-level object that attaches a key-value pair of business metadata directly
to supported Snowflake objects such as tables, columns, schemas, databases, and
warehouses.
Once a tag is set, you can query
**[SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES](https://docs.snowflake.com/en/sql-reference/account-usage/tag_references)**
(the Account Usage view that inventories direct tag associations after its
documented latency)
or call
**[SYSTEM$GET_TAG](https://docs.snowflake.com/en/sql-reference/functions/system_get_tag)**
(the scalar function that reads the effective tag value on a single object) to ask
questions like "show me this table's effective domain label" from current Snowflake
metadata.
Tags also support **[tag inheritance](https://docs.snowflake.com/en/user-guide/object-tagging#tag-lineage)**:
values assigned to supported higher-level objects propagate through the securable
object hierarchy unless overridden at a lower level. In this section we create
two governance tags — `TAG_DOMAIN` and `TAG_ENVIRONMENT` — then apply them to
`CUSTOMER_PII_TABLE`, the central table for the rest of this HOL. The same pattern
can support tag-based masking later: a compatible masking policy can be assigned
to a user-defined tag with `ALTER TAG`.


> **Outcome:** make governed objects queryable by business metadata, not just name.

### Core demo — Give sensitive data business context

A **[tag](https://docs.snowflake.com/en/user-guide/object-tagging/introduction)** is a schema-level key-value object. We will create one synthetic customer table, apply governed domain and environment labels, and read the effective values back immediately.

**You should see:** realistic fictional customer rows and effective tags of `CUSTOMER` and `HOL_DEMO`.

> Docs: [Tag inheritance](https://docs.snowflake.com/en/user-guide/object-tagging/inheritance) · [SYSTEM$GET_TAG](https://docs.snowflake.com/en/sql-reference/functions/system_get_tag)

In [ ]:
-- Core path: pre-stage representative data and apply reusable business metadata
USE ROLE SYSADMIN;
USE DATABASE GOVERNANCE_AT_SCALE_HOL;
USE SCHEMA PUBLIC;

CREATE OR REPLACE TABLE CUSTOMER_PII_TABLE (
  customer_id NUMBER, full_name VARCHAR, email VARCHAR,
  phone_number VARCHAR, ssn_last4 VARCHAR, region VARCHAR, signup_date DATE
);
INSERT INTO CUSTOMER_PII_TABLE VALUES
  (1001,'Alice Hartman','alice.hartman@example.com','555-0101','4821','US','2022-03-15'),
  (1002,'Bruno Ferreira','b.ferreira@example.net','555-0102','7743','EU','2022-05-20'),
  (1003,'Chen Wei','chenwei@example.org','555-0103','1192','APAC','2022-07-08'),
  (1004,'Diana Okafor','diana.okafor@example.com','555-0104','3367','EU','2022-08-14'),
  (1005,'Ethan Blake','ethan.blake@example.net','555-0105','9054','US','2022-09-01'),
  (1006,'Priya Sharma','priya.sharma@example.com','555-0116','4457','APAC','2023-09-02');

CREATE OR REPLACE TAG TAG_DOMAIN
  ALLOWED_VALUES 'CUSTOMER','FINANCE','OPERATIONS';
CREATE OR REPLACE TAG TAG_ENVIRONMENT
  ALLOWED_VALUES 'PRODUCTION','STAGING','HOL_DEMO';

ALTER TABLE CUSTOMER_PII_TABLE
  SET TAG TAG_DOMAIN = 'CUSTOMER', TAG_ENVIRONMENT = 'HOL_DEMO';

In [ ]:
-- Confirm: metadata and data are connected on one governed object
SELECT
  SYSTEM$GET_TAG('GOVERNANCE_AT_SCALE_HOL.PUBLIC.TAG_DOMAIN',
                 'GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE','table') AS domain_tag,
  SYSTEM$GET_TAG('GOVERNANCE_AT_SCALE_HOL.PUBLIC.TAG_ENVIRONMENT',
                 'GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE','table') AS environment_tag;

### Take-home — Scale tagging beyond one table

The recording uses two labels and immediate `SYSTEM$GET_TAG` verification. After the session, extend the pattern with cost-center tags, hierarchy overrides, `TAG_REFERENCES` inventory queries, and automatic propagation where your edition and governance model support it.

> Docs: [Work with object tags](https://docs.snowflake.com/en/user-guide/object-tagging/work) · [TAG_REFERENCES](https://docs.snowflake.com/en/sql-reference/account-usage/tag_references)

### Browse GOVERNANCE_AT_SCALE_HOL in Snowsight (read-only)

[Open in Snowsight](https://app.snowflake.com/_deeplink/#/data/databases/GOVERNANCE_AT_SCALE_HOL)

Read-only — confirm what you just built; all objects were created by the SQL above.

> **Apply on your account**
>
> In production, replace `HOL_DEMO` with `PRODUCTION` and use database- or
> schema-level tags as broad defaults where inheritance supports the child object.
> Apply a direct lower-level value only for a deliberate exception.
>
> For `TAG_COST_CENTER`, align the `ALLOWED_VALUES` with your Finance team's actual
> cost-center codes, and deploy assignments consistently so equivalent concepts do
> not fragment into incompatible values.
>
> To attach a masking policy to a user-defined tag, use
> `ALTER TAG <tag> SET MASKING POLICY <policy>`. When policy and column data types
> match, tagged columns are protected automatically. Use immediate system or
> Information Schema functions for deployment-time checks and delayed Account Usage
> views for account-wide audits.

## Section 3 — Data Classification (Horizon Catalog Auto-Tag)

**The problem:** You tagged CUSTOMER_PII_TABLE by hand in Section 2 because you already knew
which columns held sensitive data — but in a real account with hundreds of
tables added by dozens of teams, no single person can track every sensitive
column, and manual tagging only covers what someone already knows to look for.


> ⚠️ **Enterprise Edition required.** Data classification is an Enterprise+ feature.
> Confirm your account edition in Snowsight (Admin → Account) before running these steps.

Snowflake's **[sensitive data classification](https://docs.snowflake.com/en/user-guide/classify-intro)**
samples table data and recommends semantic categories for columns identified as
sensitive. Native categories include email addresses, phone numbers, national
identifiers, names, birth dates, and IP addresses. Review the results rather than
assuming that every column will receive a category. The scan is invoked via
[`SYSTEM$CLASSIFY`](https://docs.snowflake.com/en/sql-reference/stored-procedures/system_classify),
a stored procedure called with a `CALL` statement that accepts the fully-qualified
table name and a JSON options object. With `auto_tag: true`, it applies the system
tags `SNOWFLAKE.CORE.PRIVACY_CATEGORY` and `SNOWFLAKE.CORE.SEMANTIC_CATEGORY` directly
to each detected column — no per-column `ALTER TABLE … MODIFY COLUMN` required.

Each column Snowflake identifies as sensitive receives two labels: a **semantic category**
(the specific pattern, e.g. `EMAIL`, `PHONE_NUMBER`, `NATIONAL_IDENTIFIER`, `NAME`,
`DATE_OF_BIRTH`, with country-specific details such as `US_SSN`)
and a **[privacy category](https://docs.snowflake.com/en/user-guide/classify-native)** —
`IDENTIFIER` (data that can identify individuals or entities, such as email or a
national identifier), `QUASI_IDENTIFIER` (not uniquely identifying on its own,
but potentially re-identifying when combined with other data, such as postal code,
birth date, or gender), or `SENSITIVE` (confidential or private details that do not
directly identify an individual, such as salary or medical data). These labels do
not replace an assessment of applicable legal and protection requirements.

For ongoing coverage as new tables arrive, you attach a
**[classification profile](https://docs.snowflake.com/en/user-guide/classify-auto)** —
an instance of the `SNOWFLAKE.DATA_PRIVACY.CLASSIFICATION_PROFILE` class — to a
database or schema; the profile controls when tables become eligible for initial
classification and reclassification. Set `maximum_classification_validity_days` for repeat scans;
if that key is omitted, objects are never reclassified. Do not treat eligibility as a completion-time guarantee.
Those system tags create a governed inventory that can be reviewed before enforcement.
In this lab, classification identifies and labels the sensitive columns first; the next
section then applies an explicit masking policy to the approved target column. A
production design can separately attach compatible masking policies to tags.


> **Outcome:** find and label sensitive columns automatically instead of by memory.

### Core demo — Find sensitive columns before enforcing policy

**[Sensitive data classification](https://docs.snowflake.com/en/user-guide/classify-intro)** samples table data and recommends semantic and privacy categories. We pre-stage the scan because it can take longer than a live speaking beat, then inspect the stored result, including whether recommended system tags were applied.

**Important:** classification labels data; it does not prove that this lab's directly attached masking policy is active. Enforcement is the next explicit step.

> Docs: [SYSTEM$CLASSIFY](https://docs.snowflake.com/en/sql-reference/stored-procedures/system_classify) · [Classification results](https://docs.snowflake.com/en/sql-reference/functions/system_get_classification_result)

In [ ]:
-- PRE-STAGE BEFORE RECORDING: run once, then cut the wait
USE ROLE SYSADMIN;
USE DATABASE GOVERNANCE_AT_SCALE_HOL;
USE SCHEMA PUBLIC;
CALL SYSTEM$CLASSIFY(
  'GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE',
  {'auto_tag': true, 'sample_count': 1000}
);

In [ ]:
-- Inspect: review what the pre-staged scan discovered and labeled
SELECT SYSTEM$GET_CLASSIFICATION_RESULT(
  'GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE'
) AS classification_result;

### Take-home — Continuous and customized classification

The live path uses one pre-staged table scan. After the session, evaluate classification profiles, review-before-auto-tag workflows, tag maps, custom classifiers, and schema-wide classification. Validate edition, privileges, scheduling, and expected classifications in your own account before production use.

> Docs: [Use SQL to set up sensitive data classification](https://docs.snowflake.com/en/user-guide/classify-auto)

> **Apply on your account**
>
> **Apply on your account**
>
> 1. **Run `SYSTEM$CLASSIFY(..., {'auto_tag': true})` with the schema owner.**
>    That is the documented requirement for automatic system-tag application.
>    A different execution role needs the applicable object access and tag
>    privileges. Grant shared-SNOWFLAKE metadata access only to the roles that
>    actually query Account Usage or system-tag references.
>
> 2. **Enterprise Edition is required for all classification features.** This is a
>    natural driver for the Enterprise tier conversation with customers on Standard
>    Edition — classification is a flagship Horizon Catalog capability.
>
> 3. **Use SYSTEM$CLASSIFY for ad-hoc audits; use a CLASSIFICATION_PROFILE for
>    ongoing coverage.** Single-table scans are good for "I just discovered this
>    table." Profiles are what scale — they run automatically as new tables appear.
>
> 4. **Review with SYSTEM$GET_CLASSIFICATION_RESULT before enabling auto_tag in
>    production.** Call `SYSTEM$CLASSIFY` without `auto_tag` first to see what
>    Snowflake would tag. Review the JSON, then enable `auto_tag: true` once
>    satisfied with detection accuracy.
>
> 5. **Add a tag map to bridge system tags to your domain tags.** The classification
>    profile `tag_map` option maps `SNOWFLAKE.CORE.SEMANTIC_CATEGORY = 'EMAIL'` to
>    your own domain tag (e.g. `pii_tag = 'Confidential'`) automatically — existing
>    tag-based masking policies protect the user-defined tagged column when the policy
>    and column data types match.
>
> 6. **Build custom classifiers for internal terminology.** If your schema uses
>    domain-specific names not covered by Snowflake's built-in patterns (e.g.
>    `MEMBER_ID`, `TRANSACTION_HASH`), create a custom classifier and pass it via
>    `custom_classifiers` in the call options or the profile's classifier map.

## Section 4 — Column-Level Security (Masking Policies)

**The problem:** Every new audience that needs a different view of a PII table — support sees email, analysts
see nothing, developers see a hash — becomes another view to build, another permission layer to
maintain, and eventually another data copy to secure: three attack surfaces instead of one.


A **[masking policy](https://docs.snowflake.com/en/user-guide/security-column-ddm-intro)** is a
schema-level security object you attach once to a column; Snowflake evaluates it at query time and
dynamically decides what each role sees — full value, partial reveal, or total redaction — with no
extra views or table copies. The policy body is a SQL CASE expression over `CURRENT_ROLE()`:
authorized roles get `val` (the real data), everyone else gets a safe substitute such as a
`regexp_replace` that strips the personal prefix from an email address, leaving only the domain.
This is the column-level complement to the **RBAC** hierarchy and **object tags** built in the
previous sections — one physical table, multiple logical views enforced automatically at every
client and query tool.

> ⚠️ **Enterprise Edition required.** Dynamic data masking is an Enterprise+ feature. Confirm your
account edition in Snowsight (Admin → Account) before running these steps.
Source: [Understanding Dynamic Data Masking](https://docs.snowflake.com/en/user-guide/security-column-ddm-intro)


> **Outcome:** protect sensitive columns without duplicating the table per audience.

### Core demo — Enforce what each role can see

A **[masking policy](https://docs.snowflake.com/en/user-guide/security-column-ddm-intro)** is attached to a column and evaluated at query time. We will protect `email`, then run the same query as an authorized role and a comparison role.

**You should see:** full email addresses for `HOL_ACCESS_ROLE` and redacted local parts for `PUBLIC`.

> Docs: [CREATE MASKING POLICY](https://docs.snowflake.com/en/sql-reference/sql/create-masking-policy) · [Apply a masking policy](https://docs.snowflake.com/en/sql-reference/sql/alter-table-column)

In [ ]:
-- Core path: define, attach, and grant only what the comparison needs
USE ROLE SECURITYADMIN;
GRANT USAGE ON DATABASE GOVERNANCE_AT_SCALE_HOL TO ROLE SECURITYADMIN;
GRANT USAGE ON SCHEMA GOVERNANCE_AT_SCALE_HOL.PUBLIC TO ROLE SECURITYADMIN;
GRANT CREATE MASKING POLICY ON SCHEMA GOVERNANCE_AT_SCALE_HOL.PUBLIC TO ROLE SECURITYADMIN;

CREATE OR REPLACE MASKING POLICY GOVERNANCE_AT_SCALE_HOL.PUBLIC.MASK_PII_POLICY
  AS (val STRING) RETURNS STRING ->
    CASE WHEN CURRENT_ROLE() = 'HOL_ACCESS_ROLE'
         THEN val ELSE REGEXP_REPLACE(val, '.+\\@', '*****@') END;

-- Delegate only this lab policy to the table owner; avoid a persistent
-- account-level APPLY MASKING POLICY grant on a system role.
GRANT APPLY ON MASKING POLICY GOVERNANCE_AT_SCALE_HOL.PUBLIC.MASK_PII_POLICY
  TO ROLE SYSADMIN;

USE ROLE SYSADMIN;
ALTER TABLE GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE
  MODIFY COLUMN email
  SET MASKING POLICY GOVERNANCE_AT_SCALE_HOL.PUBLIC.MASK_PII_POLICY;

USE ROLE SECURITYADMIN;
-- Keep database-scoped privileges on the database role created in the RBAC section.
-- HOL_ACCESS_ROLE inherits them through:
-- GOVERNANCE_AT_SCALE_HOL.HOL_DB_ROLE -> HOL_ACCESS_ROLE.
GRANT USAGE ON SCHEMA GOVERNANCE_AT_SCALE_HOL.PUBLIC
  TO DATABASE ROLE GOVERNANCE_AT_SCALE_HOL.HOL_DB_ROLE;
GRANT SELECT ON TABLE GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE
  TO DATABASE ROLE GOVERNANCE_AT_SCALE_HOL.HOL_DB_ROLE;
GRANT USAGE ON WAREHOUSE GOVERNANCE_AT_SCALE_WH TO ROLE HOL_ACCESS_ROLE;
SET hol_user = (SELECT CURRENT_USER());
GRANT ROLE HOL_ACCESS_ROLE TO USER IDENTIFIER($hol_user);

-- PUBLIC is used only as a compact comparison role for this lab.
GRANT USAGE ON DATABASE GOVERNANCE_AT_SCALE_HOL TO ROLE PUBLIC;
GRANT USAGE ON SCHEMA GOVERNANCE_AT_SCALE_HOL.PUBLIC TO ROLE PUBLIC;
GRANT SELECT ON TABLE GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE TO ROLE PUBLIC;
GRANT USAGE ON WAREHOUSE GOVERNANCE_AT_SCALE_WH TO ROLE PUBLIC;

In [ ]:
USE ROLE HOL_ACCESS_ROLE;
USE WAREHOUSE GOVERNANCE_AT_SCALE_WH;
SELECT customer_id, email FROM GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE
ORDER BY customer_id LIMIT 4;

In [ ]:
-- Same query, different role, different visible value
USE ROLE PUBLIC;
USE WAREHOUSE GOVERNANCE_AT_SCALE_WH;
SELECT customer_id, email FROM GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE
ORDER BY customer_id LIMIT 4;

### Take-home — Production masking patterns

The recording protects one string column. After the session, create a dedicated policy-admin role, use hierarchy-aware role checks where required, centralize reusable policies, test every branch, and evaluate tag-based masking as a separate production pattern.

> Docs: [Using dynamic data masking](https://docs.snowflake.com/en/user-guide/security-column-ddm-use) · [Tag-based masking policies](https://docs.snowflake.com/en/user-guide/tag-based-masking-policies)

> **Apply on your account**
>
> In production, create a dedicated `MASKING_ADMIN` role (run by your security or privacy officer),
> grant it `CREATE MASKING POLICY ON SCHEMA` and `APPLY MASKING POLICY ON ACCOUNT`, and assign
> users there — rather than using SECURITYADMIN directly. Replace bare `CURRENT_ROLE()` checks
> with `IS_ROLE_IN_SESSION('<role>')` so users who inherit HOL_ACCESS_ROLE through a parent role
> in the hierarchy also pass the masking check, not only users who activated it as their primary
> role. For a policy shared across many tables (e.g. every table with an `email` column), create
> the policy once in a central `SECURITY` schema and reference it from all tables — one definition
> to maintain, not one copy per table. Test both CASE branches with `USE ROLE` before promoting
> to production to confirm each returns exactly what you expect.

## Section 5 — Row-Level Security (Row Access Policies)

**The problem:** The previous section masks EMAIL; it does not implement the lab's US-only entitlement. This section adds an explicit row-access rule for REGION. Masking is applied wherever a protected column appears in a query, so do not assume masking can never affect filtering or joins.


> ⚠️ **Enterprise Edition required.** Row access policies require Enterprise Edition or higher.

A **[row access policy](https://docs.snowflake.com/en/user-guide/security-row-intro)** is a schema-level security object that attaches to one or more table columns and evaluates a SQL expression at query time — returning `TRUE` to include a row and `FALSE` to exclude it. Unlike a per-audience view, a single policy centralizes the filter rule. Paired with the **column masking** from the previous section, this controls both *which rows appear* and *what protected column values look like*.


> **Outcome:** restrict which rows a role can see, not just which columns.

### Core demo — Enforce which rows a role can reach

> ⚠️ **Enterprise Edition required.**

A **[row access policy](https://docs.snowflake.com/en/user-guide/security-row-intro)** evaluates a Boolean expression for each row. We will allow admin roles to see every region and limit `HOL_ACCESS_ROLE` to `US`.

**You should see:** all three regions as `SYSADMIN`, then only `US` for the access role.

> Docs: [CREATE ROW ACCESS POLICY](https://docs.snowflake.com/en/sql-reference/sql/create-row-access-policy) · [Attach a row access policy](https://docs.snowflake.com/en/sql-reference/sql/alter-table)

In [ ]:
-- Core path: define a safe-default row filter and attach it to REGION
USE ROLE SECURITYADMIN;
GRANT CREATE ROW ACCESS POLICY ON SCHEMA GOVERNANCE_AT_SCALE_HOL.PUBLIC TO ROLE SECURITYADMIN;

CREATE OR REPLACE ROW ACCESS POLICY GOVERNANCE_AT_SCALE_HOL.PUBLIC.ROW_ACCESS_POLICY_REGION
  AS (region_col VARCHAR) RETURNS BOOLEAN ->
    CASE
      WHEN CURRENT_ROLE() IN ('SYSADMIN','SECURITYADMIN','ACCOUNTADMIN') THEN TRUE
      WHEN CURRENT_ROLE() = 'HOL_ACCESS_ROLE' THEN region_col = 'US'
      ELSE FALSE
    END;

GRANT APPLY ON ROW ACCESS POLICY GOVERNANCE_AT_SCALE_HOL.PUBLIC.ROW_ACCESS_POLICY_REGION
  TO ROLE SYSADMIN;

USE ROLE SYSADMIN;
ALTER TABLE GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE
  ADD ROW ACCESS POLICY GOVERNANCE_AT_SCALE_HOL.PUBLIC.ROW_ACCESS_POLICY_REGION ON (region);

In [ ]:
USE ROLE SYSADMIN;
SELECT region, COUNT(*) AS visible_rows
FROM GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE
GROUP BY region ORDER BY region;

In [ ]:
-- Same aggregation; the policy removes EU and APAC before aggregation
USE ROLE HOL_ACCESS_ROLE;
USE WAREHOUSE GOVERNANCE_AT_SCALE_WH;
SELECT region, COUNT(*) AS visible_rows
FROM GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE
GROUP BY region ORDER BY region;

### Take-home — Mapping-table row security

The live policy hardcodes one region so the before-and-after result is obvious. After the session, replace the branch with a protected entitlement mapping table, add negative tests for unmapped roles, and audit attachments with `POLICY_REFERENCES`.

> Docs: [Create and test row access policies](https://docs.snowflake.com/en/user-guide/security-row-create) · [POLICY_REFERENCES](https://docs.snowflake.com/en/sql-reference/functions/policy_references)

> **Apply on your account**
>
> In production:
> - Replace the hardcoded CASE WHEN CURRENT_ROLE() branches with a **mapping table**
>   — store (role, region_filter) rows and JOIN the mapping table inside the policy
>   body. Adding a new regional team then means inserting one row, not editing the
>   policy expression.
> - For larger mappings, evaluate Snowflake's documented memoizable-function
>   pattern so repeated mapping lookups can stay centralized.
> - Compare SHOW ROW ACCESS POLICIES with POLICY_REFERENCES to investigate
>   attachment gaps. Absence alone does not prove a policy is unattached:
>   Information Schema results depend on the querying role's privileges;
>   Account Usage POLICY_REFERENCES can lag by up to 120 minutes (2 hours).

## Section 6 — Access History for Audit

**The problem:** You've applied masking policies and row access policies — but you still need evidence
of which controls were enforced, and you need to answer an audit question like "who
queried our email column in the last 30 days" without resorting to guesswork.


**[ACCESS_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/access_history)**
is an Account Usage view for object and column access history — including who ran a
query, when it ran, and which supported objects and columns were accessed. The
`policies_referenced` column records enforced masking and row access policy details
for the query. These details are stored in JSON arrays; a
**[LATERAL FLATTEN](https://docs.snowflake.com/en/sql-reference/functions/flatten)**
unpacks them into rows, letting you filter to queries that touched the `email` column
and close the audit loop on the earlier masking demo.

> **Edition requirement:** ACCESS_HISTORY requires **Enterprise Edition** or higher.

> **Latency:** ACCESS_HISTORY has a latency of up to **3 hours** after a query completes.
Events generated in this section may not appear immediately — that is normal and expected.


> **Outcome:** answer 'who touched this data and when' after the fact.

### Core demo — Confirm access and policy evaluation

**[ACCESS_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/access_history)** records object and column access plus referenced policies. Because Account Usage is delayed, generate the evidence before recording and use the live segment to inspect the landed audit rows.

**You should see:** the user and query that touched `EMAIL`, plus `MASK_PII_POLICY` and `ROW_ACCESS_POLICY_REGION` in policy evidence after latency has elapsed.

> Docs: [ACCESS_HISTORY data latency](https://docs.snowflake.com/en/sql-reference/account-usage/access_history#data-latency)

In [ ]:
-- PRE-STAGE AT LEAST THREE HOURS BEFORE RECORDING
USE ROLE SECURITYADMIN;
GRANT DATABASE ROLE SNOWFLAKE.GOVERNANCE_VIEWER TO ROLE HOL_ACCESS_ROLE;

USE ROLE HOL_ACCESS_ROLE;
USE WAREHOUSE GOVERNANCE_AT_SCALE_WH;
SELECT customer_id, email, region
FROM GOVERNANCE_AT_SCALE_HOL.PUBLIC.CUSTOMER_PII_TABLE
ORDER BY customer_id LIMIT 4;

In [ ]:
-- Live evidence: who queried the protected EMAIL column?
USE ROLE HOL_ACCESS_ROLE;
SELECT DISTINCT
  ah.query_id,
  ah.user_name,
  ah.query_start_time,
  obj.value:"objectName"::STRING AS object_name,
  col.value:"columnName"::STRING AS column_name
FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY ah,
  LATERAL FLATTEN(input => ah.direct_objects_accessed) obj,
  LATERAL FLATTEN(input => obj.value:"columns") col
WHERE ah.query_start_time >= DATEADD(day,-1,CURRENT_TIMESTAMP())
  AND obj.value:"objectName"::STRING ILIKE '%GOVERNANCE_AT_SCALE_HOL%CUSTOMER_PII_TABLE%'
  AND col.value:"columnName"::STRING = 'EMAIL'
ORDER BY ah.query_start_time DESC;

In [ ]:
-- Live confirmation: which governance policies were enforced?
SELECT DISTINCT
  ah.query_id,
  ah.user_name,
  ah.query_start_time,
  f.value:"policyName"::STRING AS policy_name,
  f.value:"policyKind"::STRING AS policy_kind
FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY ah,
  LATERAL FLATTEN(input => ah.policies_referenced, recursive => TRUE) f
WHERE ah.query_start_time >= DATEADD(day,-1,CURRENT_TIMESTAMP())
  AND f.value:"policyName"::STRING IN (
    'GOVERNANCE_AT_SCALE_HOL.PUBLIC.MASK_PII_POLICY',
    'GOVERNANCE_AT_SCALE_HOL.PUBLIC.ROW_ACCESS_POLICY_REGION'
  )
ORDER BY ah.query_start_time DESC, policy_name;

### Take-home — Operationalize the audit trail

The live path answers one column-access question and confirms two enforced policy references. After the session, expand the window, join `QUERY_HISTORY` for query text, materialize recurring reports only after validating cost and retention needs, and test the exact JSON paths in your account.

> Docs: [ACCESS_HISTORY](https://docs.snowflake.com/en/sql-reference/account-usage/access_history)

> **Apply on your account**
>
> **On your account:** Replace the `ILIKE '%GOVERNANCE_AT_SCALE_HOL%CUSTOMER_PII_TABLE%'`
> filter with your production database and table names. For a compliance-grade workflow:
>
> - Create a dedicated `AUDITOR` role and grant it `DATABASE ROLE SNOWFLAKE.GOVERNANCE_VIEWER`
>   (more targeted than broadening SYSADMIN access to Account Usage).
> - Use `DATEADD(day, -30, CURRENT_TIMESTAMP())` for a rolling 30-day compliance window.
> - Evaluate cost, retention, and freshness requirements before materializing a
>   recurring audit report with a scheduled task.
> - Join to `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` on `query_id` to add its recorded
>   `query_text` alongside each column-access record. The text is limited to 100K
>   characters; longer SQL statements are truncated.
> - `SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY` retains 365 days. If your requirement is
>   longer, archive the needed records before they age out.

## Cleanup

Cleanup is disabled by default: the following SQL is enclosed in a `/* ... */` block comment. Leave it commented during the demo and when using Run All. Only remove the opening `/*` and closing `*/` when you intentionally want to delete the lab database, warehouse, and roles. Confirm the account and object names first.

This protects the cleanup cell only. Setup and other demo cells still create or replace lab objects; review them before re-running.

In [ ]:
/* CLEANUP DISABLED BY DEFAULT. Remove the surrounding block comment only for intentional teardown.
USE ROLE SECURITYADMIN;
SET hol_user = (SELECT CURRENT_USER());
REVOKE ROLE HOL_ACCESS_ROLE FROM USER IDENTIFIER($hol_user);
USE ROLE USERADMIN;
DROP ROLE IF EXISTS HOL_ACCESS_ROLE;
DROP ROLE IF EXISTS HOL_FUNCTIONAL_ROLE;
USE ROLE SYSADMIN;
DROP DATABASE IF EXISTS GOVERNANCE_AT_SCALE_HOL;
USE ROLE SYSADMIN;
DROP WAREHOUSE IF EXISTS GOVERNANCE_AT_SCALE_WH;
*/

## Optional follow-ons from current documentation

Documentation checked on 2026-10-05. These are reading references, not additional executable demo steps. The core lab does not enable these features.

- **Role-grant delegation (October 5, 2026):** "You can now grant an account role or a database role to another role with WITH GRANT OPTION." The release note also states: "A grant to a user doesn't support the clause." [Release note](https://docs.snowflake.com/en/release-notes/2026/other/2026-10-05-role-grant-option)
- **Inherited grants and scoped grant administration (September 10, 2026):** "Inherited grants and container-level MANAGE GRANTS are now generally available." The release note qualifies this: "Both features remain opt-in." Review these as alternatives when designing grant administration; this lab does not change account feature settings. [Release note](https://docs.snowflake.com/en/release-notes/2026/other/2026-09-10-inherited-grants-ga)
- **Multi-value tags (August 25, 2026):** "With this release, multi-value tags are generally available. You can assign more than one string value to the same tag." This lab deliberately keeps its domain and environment tags single-value. The reference warns: "If you call SYSTEM$GET_TAG on an object that has multiple values for a tag, the function returns an error." [Release note](https://docs.snowflake.com/en/release-notes/2026/other/2026-08-25-multi-value-tags-ga) | [Reference](https://docs.snowflake.com/en/user-guide/object-tagging/multi-value-tags)
- **AI mode for classification (August 17, 2026, public preview):** "Sensitive data classification now supports AI mode in public preview." Treat it as a separate evaluation, not a requirement for this lab's standard classification call. [Release note](https://docs.snowflake.com/en/release-notes/2026/other/2026-08-17-sensitive-data-classification-ai-mode-preview)
- **Tag-based row access policies (preview):** "Tag-based row access policies combine the object tagging and row access policy features to allow a row access policy to be set on a tag using an ALTER TAG command." The reference labels this "Preview Feature" and warns: "If no alias tuple matches, the policy isn't enforced on that table." Keep the direct table attachment in this demo; evaluate argument-to-column matching separately before adopting tag-based attachment. [Reference](https://docs.snowflake.com/en/user-guide/tag-based-row-access-policies)
- **Agent audit context:** "The agents_info column is an ordered JSON array of invoking agents, from the nearest agent to the top-level agent." The release note states: "This column is NULL when no agent is involved in the object access." This SQL-only demo does not generate agent activity. [Release note](https://docs.snowflake.com/en/release-notes/2026/other/2026-08-03-access-history-agents-info-column)

These references do not certify an end-to-end lab run. See `DEMO_REVIEW_2026-10-05.md` for fixes and outstanding validation.